# Extra · Anomalies and the seasonal cycle

How much has the climate warmed? To see it, remove what repeats every year, the **seasonal cycle**, and look
at what is left: the **anomalies**. In this notebook you

1. compute the seasonal cycle of the global temperature for two periods,
2. compute **anomalies** against a reference period,
3. fit a **trend** to them, and
4. map the warming and its trend over the globe.

Data: ERA5 2 m temperature, monthly, 1950–2023. **Reference period**: 1950–1980; **recent period**: 1981–2023.

## 1. The global mean

The global mean is the same at every zoom (class 06), so zoom 0 (12 cells) is enough, and fast.

In [ ]:
import easygems.healpix as egh
import intake
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs

cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")
temperature = cat.ERA5(zoom=0).to_dask()["2t"].sel(time=slice("1950", "2023")).compute()
global_mean = temperature.mean("cell") - 273.15

reference = global_mean.sel(time=slice("1950", "1980"))
recent = global_mean.sel(time=slice("1981", "2023"))

## 2. The seasonal cycle

`.groupby("time.month")` puts all Januaries together, all Februaries, and so on; `.mean()` gives the mean of
each month: the **seasonal cycle**. `.std()` tells how much a month varies from year to year.

In [ ]:
month_names = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

fig, ax = plt.subplots(figsize=(8, 4))
for period, name in [(reference, "1950–1980"), (recent, "1981–2023")]:
    cycle = period.groupby("time.month").mean()
    spread = period.groupby("time.month").std()
    ax.plot(cycle.month, cycle, marker="o", label=name)
    ax.fill_between(cycle.month, cycle - spread, cycle + spread, alpha=0.3)
ax.set_xticks(range(1, 13), month_names)
ax.set_ylabel("temperature / °C")
ax.set_title("Seasonal cycle of the global mean 2 m temperature (ERA5)")
ax.legend()

fig.savefig(
    "/app/pages/anomalies_seasonal_cycle/seasonal_cycle.png",
    dpi=100,
    bbox_inches="tight",
)
plt.show()

The globe is warmest in July: there is more land in the north, and land warms faster than the ocean.
The recent period is warmer in every month.

## 3. Anomalies

An **anomaly** is a value minus the mean of the same month in the reference period. `groupby` subtracts the
right month from each value:

In [ ]:
climatology = reference.groupby("time.month").mean()
anomalies = recent.groupby("time.month") - climatology
yearly = anomalies.resample(time="YS").mean()

years = yearly.time.dt.year.values
slope, intercept = np.polyfit(
    years, yearly.values, 1
)  # a straight line: slope per year
print(f"trend: {10 * slope:.2f} °C per decade")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(years, yearly, color=np.where(yearly > 0, "firebrick", "steelblue"))
ax.plot(
    years,
    slope * years + intercept,
    color="black",
    label=f"trend: {10 * slope:.2f} °C per decade",
)
ax.set_ylabel("anomaly / °C")
ax.set_title("Global mean 2 m temperature, anomaly against 1950–1980 (ERA5)")
ax.legend()

fig.savefig(
    "/app/pages/anomalies_seasonal_cycle/anomalies.png", dpi=100, bbox_inches="tight"
)
plt.show()

## 4. Maps of the warming

The same at every cell, at zoom 6: the mean anomaly of 1981–2023, and the trend of each cell. `np.polyfit`
fits a line to every column of a table at once: here one column per cell. `.compute()` loads the data
into memory first (about 175 MB; it can take a minute): grouping by month needs it.

In [ ]:
cells = (
    cat.ERA5(zoom=6).to_dask()["2t"].sel(time=slice("1950", "2023")).compute()
)  # into memory
cells_climatology = cells.sel(time=slice("1950", "1980")).groupby("time.month").mean()
cells_yearly = (
    (cells.sel(time=slice("1981", "2023")).groupby("time.month") - cells_climatology)
    .resample(time="YS")
    .mean()
)

mean_anomaly = cells_yearly.mean("time")
trend = (
    10 * np.polyfit(cells_yearly.time.dt.year.values, cells_yearly.values, 1)[0]
)  # °C per decade

fig, axes = plt.subplots(
    1, 2, figsize=(16, 4), subplot_kw={"projection": ccrs.Robinson()}
)
for ax, data, title, limit in [
    (axes[0], mean_anomaly, "Mean anomaly 1981–2023 / °C", 2),
    (axes[1], trend, "Trend 1981–2023 / °C per decade", 1),
]:
    ax.set_global()
    ax.coastlines(linewidth=0.4)
    image = egh.healpix_show(data, ax=ax, cmap="RdBu_r", vmin=-limit, vmax=limit)
    fig.colorbar(image, ax=ax, shrink=0.7)
    ax.set_title(title)

fig.savefig(
    "/app/pages/anomalies_seasonal_cycle/anomaly_maps.png", dpi=100, bbox_inches="tight"
)
plt.show()

Where does it warm most? Look at the land compared with the ocean, and at the Arctic.

## 5. Exercise

Do sections 2 and 3 for **one place** instead of the globe, e.g. the cell of Hamburg or of your region (class 05:
`hp.ang2pix(2**zoom, lon, lat, nest=True, lonlat=True)`, and `.isel(cell=...)` instead of `.mean("cell")`;
use zoom 6). Is the trend of one place larger or smaller than the global one? Why are its anomalies more
irregular? Save your plot as `/app/pages/anomalies_seasonal_cycle/exercise.png`.

In [ ]:
# the seasonal cycle and the anomalies of one place

## Your page

Open `pages/anomalies_seasonal_cycle/my_page.rst` in JupyterLab and describe what you found. Look at it on
http://localhost:8000 → *Extras*.